# Scoring IRV

How does the original IRV data score against the more recent corrections?

Process:
* Get latest data from ClearAligner
* Stage in data/alignments
* Score with new data as reference and old data as hypothesis: this requires some non-standard parameters

In [1]:
# PARAMETERS: your local copy of alignments-eng/data
(targetlang, targetid, sourceid) = ("asm", "IRVAsm", "SBLGNT")

from biblealignlib.burrito import CLEARROOT, AlignmentSet
from biblealignlib.autoalign import scorer
# bundled parameters for downstream processes
alsetref = AlignmentSet(targetlanguage=targetlang,
        targetid=targetid,
        sourceid=sourceid,
        langdatapath=(CLEARROOT / f"alignments-{targetlang}/data"), alternateid="20250113manual")


## Instantiating a Scorer

Details: [Scoring Alignment Data > Instantiating a Scorer](31ScoringAlignmentData.ipynb#Instantiating-a-Scorer)

In [2]:
# this directory should already exist and have burrito format alignments
# non-standard: condition data is not in exp
conditiondir = (CLEARROOT / f"alignments-{targetlang}/data/alignments/IRVAsm")
print(f"Conditiondir: {conditiondir}")

sc = scorer.Scorer(referenceset=alsetref, 
                   hypothesispath = (conditiondir / f"{sourceid}-{targetid}-manual.json"), 
                   hypothesisaltid="manual")

Conditiondir: /Users/sboisen/git/Clear-Bible/alignments-asm/data/alignments/IRVAsm

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/targets/IRVAsm/nt_IRVAsm.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/alignments/IRVAsm/SBLGNT-IRVAsm-20250113manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/alignments/IRVAsm/SBLGNT-IRVAsm-20250113manual.toml
        
----- Hypothesis data: <AlignmentSet: asm, SBLGNT-IRVAsm-manual>

        - sourcepath: /Users/sboisen/git/Clear-Bible/Alignments/data/sources/SBLGNT.tsv
        - targetpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/targets/IRVAsm/nt_IRVAsm.tsv
        - alignmentpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/alignments/IRVAsm/SBLGNT-IRVAsm-manual.json
        - tomlpath: /Users/sboisen/git/Clear-Bible/alignments-asm/data/alignments/IRVAsm/S

### Score the Corpus

In [3]:
print(sc.score_all().summary())

all: AER=0.4357	P=0.5643	R=0.7664	F1=0.6500


### Visualize Scores for Gospels + Acts


In [24]:
import altair as alt
import pandas as pd

# MAT-LUK
partialscore = sc.score_partial(startbcv="40001001", endbcv="43024050")

source = pd.DataFrame([vs.asdict() for vs in partialscore.verse_scores])


/Users/sboisen/git/Clear-Bible/biblealignlib/biblealignlib/burrito/util.py:90: UserWarning: Did not stop collecting: check endbcv 43024050
  warnings.warn(f"Did not stop collecting: check endbcv {endbcv}")


In [36]:
# 5272 scores: pick the first 4995 to meet Altair limitations
allscore = sc.score_all()

source = pd.DataFrame([vs.asdict() for vs in allscore.verse_scores[:4995]])


In [37]:
range = "Most"
chart = alt.Chart(source, title=f"{range} F1 Scores ({sourceid}-{targetid})").mark_rect().encode(
    x='Verse:O',
    y='Chapter:O',
    color='F1:Q',
    tooltip=['Reference', 'F1', 'Precision', 'Recall']
)
chart.save(f'{range}.html')
chart

alt.Chart(...)

In [ ]:
# ROM-REV
partialscore = sc.score_partial(startbcv="45001001", endbcv="66022020")
